# Formula AST experiment

Load existing compositional explanation search results as written formula trees, then define each neuron Boolean as its formula in one Z3 solver.

In [ ]:
from pathlib import Path

import pandas as pd
import z3

from analysis import load_compexp, to_z3

In [ ]:
csv_path = Path("../data/snli_beam_results.csv")
results = load_compexp(csv_path)
results.head()

In [ ]:
formula_rows = results.loc[results["ast"].notna()]
formula_rows[["neuron", "formula", "iou", "ast"]].head()

In [ ]:
def formula_to_string(node):
    """Reconstruct search-format text using only the AST."""
    if node.kind == "atom":
        return node.value
    if node.kind == "constant":
        return str(node.value)
    if node.kind == "NOT":
        return f"(NOT {formula_to_string(node.children[0])})"
    if node.kind in {"AND", "OR"}:
        children = (formula_to_string(child) for child in node.children)
        return "(" + f" {node.kind} ".join(children) + ")"
    raise ValueError(f"Unknown formula node kind {node.kind!r}")


round_trip = formula_rows[["neuron", "formula"]].copy()
round_trip["reconstructed_formula"] = formula_rows["ast"].map(formula_to_string)
round_trip["matches"] = round_trip["formula"] == round_trip["reconstructed_formula"]
mismatches = round_trip.loc[~round_trip["matches"]]
assert mismatches.empty, f"AST round-trip mismatches:\n{mismatches.to_string(index=False)}"
round_trip


## Load solver constraints

Each neuron Boolean equals its formula. Feature names share Boolean variables across neurons; empty and pruned formulas are skipped.

In [ ]:
solver = z3.Solver()
for neuron_id, node in zip(formula_rows["neuron"], formula_rows["ast"]):
    neuron = z3.Bool(f"neuron_{neuron_id}")
    solver.add(neuron == to_z3(node))

len(solver.assertions())

## Enumerate valid neuron patterns

Record each satisfying neuron assignment, then require at least one neuron to differ. Stop at `unsat`. Feature assignments that produce the same neuron pattern are counted once. Only neurons with usable formulas participate.

In [ ]:
neuron_ids = formula_rows["neuron"].tolist()
neuron_variables = [z3.Bool(f"neuron_{neuron_id}") for neuron_id in neuron_ids]
enumeration_solver = z3.Solver()
enumeration_solver.add(*solver.assertions())
patterns = []

while True:
    status = enumeration_solver.check()
    if status == z3.unsat:
        break
    if status != z3.sat:
        raise RuntimeError(f"Enumeration incomplete: {enumeration_solver.reason_unknown()}")

    model = enumeration_solver.model()
    values = [model.eval(neuron, model_completion=True) for neuron in neuron_variables]
    patterns.append(tuple(z3.is_true(value) for value in values))
    print(len(patterns), sorted(
        neuron_id for neuron_id, active in zip(neuron_ids, patterns[-1]) if active
    ))
    enumeration_solver.add(z3.Or([
        neuron != value for neuron, value in zip(neuron_variables, values)
    ]))

activation_patterns = pd.DataFrame(patterns, columns=neuron_ids, dtype=bool)
activation_patterns
